# Notebook Dedicated to Construct the Filtered file.dat to Smartly Download RATDS from Grid Server

In [7]:
import numpy as np
import re
import uproot
import os
import pickle

# Create Resumed Data List to Download from Grid

The idea is to take the file.dat (in txt format) and read all the entries and compare the runID and subrunID with the candidates. Then, only save the lines of file.dat that contains the run and subrun of candidates.

## Filtering Function

In [6]:
def filter_filelist(in_file_dir, out_file_dir, candidate_list):
    
    """
    Function designed to read the txt file of the dat file and return
    the filtered lines that corresponde to the run and subrun of the candidates

    Parameters:
    - in_file_dir: Directory and name of the file.dat
    - out_file_dir: directory and name of the output file 
    - candidate_list: Set of tuples (runID, subrunID) of the candidates.
    """
    
    # Usamos un set para que la búsqueda sea instantánea O(1)
    candidate_set = set(candidate_list)
    lines_found = 0

    # Expresión regular para capturar run (r...) y subrun (s...)
    # El patrón busca '_r' seguido de dígitos y '_s' seguido de dígitos
    patron = re.compile(r'_r(\d+)_s(\d+)_')

    with open(in_file_dir, 'r') as f_in, open(out_file_dir, 'w') as f_out:
        for linea in f_in:
            # El nombre del archivo es la primera columna (separada por tabulador)
            filename = linea.split('\t')[0]
            #print(f'file name: {filename}')
            
            match = patron.search(filename)
            if match:
                # Convertimos a int para ignorar los ceros a la izquierda (0000366261 -> 366261)
                run_id = int(match.group(1))
                subrun_id = int(match.group(2))
                #print(f'run {run_id}')
                #print(f'subrun {subrun_id}')
                
                # Comprobamos si este par (run, subrun) está en tus candidatos
                if (run_id, subrun_id) in candidate_set:
                    f_out.write(linea)
                    lines_found += 1
                    
    print(f"Proceso finalizado. Se encontraron {lines_found} archivos coincidentes.")

# Load and Select Data
Load runID and subrunID of Candidates
Load observables to perform cuts

In [13]:
# ------- Observable list -------
obs_list = ['energy_corrected', 'posr_av', 'itr', 'eventID', 'runID', 'subrunID']

# ------- Observable Dictionary -------
obs_dict = {obs: np.array([]) for obs in obs_list}

# ------- Directory of data -------
candidates_data_dir = 'E:/Data/solars/solarnu_Realdata/2p2PPO/resume_files/'
hs_dir = 'E:/Data/solars/solarnu_Realdata/2p2PPO/HS_results/pkl_resume/'

# Loop over the observable list to load the candidates data in obs_dict
for obs in obs_list:
    
    obs_dir = candidates_data_dir + obs + '.npy'
    obs_i = np.load(obs_dir)
    obs_dict[obs] = np.append(obs_dict[obs], obs_i)

print(f'Nº of intial events : {len(obs_dict['energy_corrected'])}')

# ===== Apply General Cuts =====
en_inf_cut = 5
posr_cut = 5500

energy_condition = (obs_dict['energy_corrected'] >= en_inf_cut)
posr_condition = (obs_dict['posr_av'] <= posr_cut)
runID_condition = (obs_dict['runID'] > 301000)
itr_condition = (obs_dict['itr'] >= 0.20)

mask = (energy_condition & posr_condition & runID_condition & itr_condition)

energy = obs_dict['energy_corrected'][mask]
posr_av = obs_dict['posr_av'][mask]
runID = obs_dict['runID'][mask]
subrunID = obs_dict['subrunID'][mask]
itr = obs_dict['itr'][mask]
eventID = obs_dict['eventID'][mask]

print(f'Nº of events after basic cuts: {len(energy)}')

# ===== Now Apply Hotspot Cuts =====

# Load HS results
with open(hs_dir + 'hs_prompt_resume.pkl', 'rb') as f:
    hs_prompt_dict = pickle.load(f)
    
    hs_prompt_eventID = hs_prompt_dict['eventID']
    hs_prompt_runID = hs_prompt_dict['runID']
    
with open(hs_dir + 'hs_delay_resume.pkl', 'rb') as f:
    hs_delay_dict = pickle.load(f)
    
    hs_delay_eventID = hs_delay_dict['eventID']
    hs_delay_runID = hs_delay_dict['runID']

hs_eventID = np.concatenate((hs_prompt_eventID, hs_delay_eventID))
hs_runID = np.concatenate((hs_prompt_runID, hs_delay_runID))

# Remove coincident eventID and runID events with tagged HS eventID and runID
# Create an unique number such that runID*offset + eventID is an unique number

offset = np.int64(10**10)

unique_id_data = (runID.astype(np.int64) * offset) + eventID.astype(np.int64)
unique_id_hs = (hs_runID.astype(np.int64) * offset) + hs_eventID.astype(np.int64)

is_hotspot = np.isin(unique_id_data, unique_id_hs)

hs_cut = ~is_hotspot

# Select the data
energy = energy[hs_cut]
posr_av = posr_av[hs_cut]
runID = runID[hs_cut]
subrunID = subrunID[hs_cut]
itr = itr[hs_cut]
eventID = eventID[hs_cut]

print(f'Nº of events after removing HS events: {len(energy)}')

Nº of intial events : 33543
Nº of events after basic cuts: 174
Nº of events after removing HS events: 173


## Start Filtering File Names

In [15]:
in_file_dir = 'dat_files_for_RATDS/raw/Analysis20_PPOR_801_802_Bronze_300000_310637_RAL_ratds.dat'
out_file_dir = 'dat_files_for_RATDS/filtered/Analysis20_PPOR_801_802_Bronze_300000_310637_RAL_ratds_solar_filtered.dat'
candidate_list = list(zip(runID, subrunID))

filter_filelist(in_file_dir, out_file_dir, candidate_list)

Proceso finalizado. Se encontraron 172 archivos coincidentes.


In [16]:
candidate_list

[(301383.0, 8.0),
 (301418.0, 3.0),
 (301475.0, 8.0),
 (301480.0, 1.0),
 (301494.0, 6.0),
 (301579.0, 8.0),
 (301649.0, 0.0),
 (301653.0, 0.0),
 (301721.0, 6.0),
 (301783.0, 4.0),
 (301973.0, 6.0),
 (302003.0, 10.0),
 (302005.0, 3.0),
 (302357.0, 0.0),
 (302381.0, 4.0),
 (302405.0, 3.0),
 (302410.0, 12.0),
 (302419.0, 1.0),
 (302596.0, 5.0),
 (302760.0, 12.0),
 (303039.0, 1.0),
 (303109.0, 10.0),
 (303443.0, 4.0),
 (303651.0, 10.0),
 (303666.0, 6.0),
 (303858.0, 1.0),
 (303878.0, 10.0),
 (303883.0, 1.0),
 (304091.0, 3.0),
 (304095.0, 9.0),
 (304120.0, 13.0),
 (304159.0, 9.0),
 (304171.0, 4.0),
 (304191.0, 8.0),
 (304199.0, 6.0),
 (304205.0, 10.0),
 (304256.0, 4.0),
 (304297.0, 10.0),
 (304436.0, 4.0),
 (304444.0, 4.0),
 (304449.0, 3.0),
 (304526.0, 9.0),
 (304534.0, 2.0),
 (304544.0, 3.0),
 (304551.0, 5.0),
 (304670.0, 6.0),
 (304709.0, 4.0),
 (304713.0, 4.0),
 (304846.0, 0.0),
 (304896.0, 11.0),
 (304913.0, 2.0),
 (305059.0, 9.0),
 (305087.0, 12.0),
 (305366.0, 12.0),
 (305386.0, 0.0)